## Imports and load data

In [23]:
import json
import joblib
import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import List, Dict, Tuple

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, classification_report,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance

pd.set_option("display.max_columns", 200)
np.random.seed(42)


In [24]:
# Update path to your file
df = pd.read_csv("data/raw/ai4i.csv")
df.head()


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## Basic data checks (schema + missingness)

In [25]:
expected_cols = {
    "UDI","Product ID","Type",
    "Air temperature [K]","Process temperature [K]",
    "Rotational speed [rpm]","Torque [Nm]","Tool wear [min]",
    "Machine failure","TWF","HDF","PWF","OSF","RNF"
}

missing = expected_cols - set(df.columns)
extra = set(df.columns) - expected_cols

print("Missing expected columns:", missing)
print("Extra columns:", extra)

print("\nNull counts:\n", df.isna().sum().sort_values(ascending=False).head(15))
print("\nTarget prevalence:\n", df["Machine failure"].value_counts(normalize=True))



Missing expected columns: set()
Extra columns: set()

Null counts:
 UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

Target prevalence:
 Machine failure
0    0.9661
1    0.0339
Name: proportion, dtype: float64


## Leakage-safe feature selection + engineered features

In [26]:
TARGET = "Machine failure"

# Leakage / identifiers to exclude from features
LEAK_OR_ID = ["UDI", "Product ID", "TWF", "HDF", "PWF", "OSF", "RNF", TARGET]

base_cat = ["Type"]
base_num = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

X = df.drop(columns=[c for c in LEAK_OR_ID if c in df.columns]).copy()
y = df[TARGET].astype(int)

# Add physically meaningful engineered features
X["TempDiff"] = df["Process temperature [K]"] - df["Air temperature [K]"]      # process-air delta
X["Torque_x_RPM"] = df["Torque [Nm]"] * df["Rotational speed [rpm]"]           # proxy of mechanical power

cat_cols = base_cat
num_cols = base_num + ["TempDiff", "Torque_x_RPM"]

X = X[cat_cols + num_cols]

X.head(), y.value_counts()


(  Type  Air temperature [K]  Process temperature [K]  Rotational speed [rpm]  \
 0    M                298.1                    308.6                    1551   
 1    L                298.2                    308.7                    1408   
 2    L                298.1                    308.5                    1498   
 3    L                298.2                    308.6                    1433   
 4    L                298.2                    308.7                    1408   
 
    Torque [Nm]  Tool wear [min]  TempDiff  Torque_x_RPM  
 0         42.8                0      10.5       66382.8  
 1         46.3                3      10.5       65190.4  
 2         49.4                5      10.4       74001.2  
 3         39.5                7      10.4       56603.5  
 4         40.0                9      10.5       56320.0  ,
 Machine failure
 0    9661
 1     339
 Name: count, dtype: int64)

## Build pipeline + train modelTrain/test split (stratified)

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape, " Test size:", X_test.shape)
print("Train prevalence:", y_train.mean(), " Test prevalence:", y_test.mean())


Train size: (8000, 8)  Test size: (2000, 8)
Train prevalence: 0.033875  Test prevalence: 0.034


## Preprocessing pipeline (categorical + numeric)

In [28]:
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", Pipeline([("scaler", StandardScaler())]), num_cols),
    ],
    remainder="drop"
)


## Baseline model (Logistic Regression, class-weighted)

In [30]:
lr = LogisticRegression(max_iter=3000, class_weight="balanced")

lr_clf = Pipeline(steps=[
    ("prep", preprocess),
    ("model", lr)
])

lr_clf.fit(X_train, y_train)

proba_lr = lr_clf.predict_proba(X_test)[:, 1]

print("LR PR-AUC:", average_precision_score(y_test, proba_lr))
print("LR ROC-AUC:", roc_auc_score(y_test, proba_lr))




LR PR-AUC: 0.4658798217383025
LR ROC-AUC: 0.933952928997686


## Stronger model (HistGradientBoosting)

In [32]:
hgb = HistGradientBoostingClassifier(
    random_state=42,
    max_depth=6,
    learning_rate=0.05,
    max_iter=300
)

hgb_clf = Pipeline(steps=[
    ("prep", preprocess),
    ("model", hgb)
])

hgb_clf.fit(X_train, y_train)

proba_hgb = hgb_clf.predict_proba(X_test)[:, 1]

print("HGB PR-AUC:", average_precision_score(y_test, proba_hgb))
print("HGB ROC-AUC:", roc_auc_score(y_test, proba_hgb))



HGB PR-AUC: 0.8927587272579237
HGB ROC-AUC: 0.9686396297649493


## Operational metric: Precision@k / Recall@k (maintenance triage)

In [33]:
def precision_recall_at_k(y_true: pd.Series, scores: np.ndarray, k: int) -> Tuple[float, float]:
    idx = np.argsort(-scores)[:k]
    precision_k = float(y_true.iloc[idx].mean())
    recall_k = float(y_true.iloc[idx].sum() / max(y_true.sum(), 1))
    return precision_k, recall_k

for k in [10, 20, 50, 100]:
    p_k, r_k = precision_recall_at_k(y_test.reset_index(drop=True), proba_hgb, k)
    print(f"k={k:3d}  precision@k={p_k:.3f}  recall@k={r_k:.3f}")


k= 10  precision@k=1.000  recall@k=0.147
k= 20  precision@k=1.000  recall@k=0.294
k= 50  precision@k=1.000  recall@k=0.735
k=100  precision@k=0.600  recall@k=0.882


## max threshold selection

In [34]:
prec, rec, thr = precision_recall_curve(y_test, proba_hgb)

f1 = 2 * (prec * rec) / (prec + rec + 1e-12)

# Align: thr length = len(prec)-1
best_i = np.argmax(f1[1:]) + 1
best_thr_f1 = float(thr[best_i - 1])

best_thr_f1


0.6780237743237058

## Capacity-based threshold (top-20/day)

In [36]:
def topk_threshold(scores: np.ndarray, k: int) -> float:
    s = np.sort(scores)[::-1]
    return float(s[k-1])

best_thr_top20 = topk_threshold(proba_hgb, k=20)
best_thr_top20


0.9908855692071712

## Report classification metrics at your chosen threshold

In [37]:
def report_at_threshold(y_true: pd.Series, scores: np.ndarray, threshold: float) -> Dict:
    pred = (scores >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred)
    rep = classification_report(y_true, pred, output_dict=True)
    return {"threshold": threshold, "confusion_matrix": cm, "report": rep}

out_f1 = report_at_threshold(y_test, proba_hgb, best_thr_f1)
out_top20 = report_at_threshold(y_test, proba_hgb, best_thr_top20)

print("=== F1 threshold ===")
print("thr:", out_f1["threshold"])
print("confusion:\n", out_f1["confusion_matrix"])
print(pd.DataFrame(out_f1["report"]).T[["precision","recall","f1-score","support"]])

print("\n=== Top-20 threshold ===")
print("thr:", out_top20["threshold"])
print("confusion:\n", out_top20["confusion_matrix"])
print(pd.DataFrame(out_top20["report"]).T[["precision","recall","f1-score","support"]])


=== F1 threshold ===
thr: 0.6780237743237058
confusion:
 [[1930    2]
 [  15   53]]
              precision    recall  f1-score    support
0              0.992288  0.998965  0.995615  1932.0000
1              0.963636  0.779412  0.861789    68.0000
accuracy       0.991500  0.991500  0.991500     0.9915
macro avg      0.977962  0.889188  0.928702  2000.0000
weighted avg   0.991314  0.991500  0.991065  2000.0000

=== Top-20 threshold ===
thr: 0.9908855692071712
confusion:
 [[1932    0]
 [  48   20]]
              precision    recall  f1-score   support
0              0.975758  1.000000  0.987730  1932.000
1              1.000000  0.294118  0.454545    68.000
accuracy       0.976000  0.976000  0.976000     0.976
macro avg      0.987879  0.647059  0.721138  2000.000
weighted avg   0.976582  0.976000  0.969602  2000.000


## Probability calibration (recommended for “risk scores” in an app)

In [38]:
# Calibrate the whole pipeline using CV on training data only
calibrated = CalibratedClassifierCV(
    estimator=hgb_clf,
    method="isotonic",
    cv=3
)
calibrated.fit(X_train, y_train)

proba_cal = calibrated.predict_proba(X_test)[:, 1]

print("Calibrated PR-AUC:", average_precision_score(y_test, proba_cal))
print("Calibrated ROC-AUC:", roc_auc_score(y_test, proba_cal))

for k in [20, 50]:
    p_k, r_k = precision_recall_at_k(y_test.reset_index(drop=True), proba_cal, k)
    print(f"[Calibrated] k={k} precision@k={p_k:.3f} recall@k={r_k:.3f}")


Calibrated PR-AUC: 0.9069876185020286
Calibrated ROC-AUC: 0.9791438314456217
[Calibrated] k=20 precision@k=1.000 recall@k=0.294
[Calibrated] k=50 precision@k=1.000 recall@k=0.735


## Interpretability (permutation importance on test set)

In [39]:
# Use the calibrated model for explanations (or hgb_clf if you prefer)
model_for_pi = calibrated

pi = permutation_importance(
    model_for_pi, X_test, y_test,
    n_repeats=8, random_state=42, scoring="average_precision"
)

imp = pd.Series(pi.importances_mean, index=X_test.columns).sort_values(ascending=False)
imp


Rotational speed [rpm]     0.528379
TempDiff                   0.445738
Tool wear [min]            0.280600
Torque_x_RPM               0.159337
Torque [Nm]                0.107829
Type                       0.022132
Process temperature [K]    0.000675
Air temperature [K]       -0.003324
dtype: float64

## Single-row “app-like” prediction

In [40]:
sample = {
    "Type": "M",
    "Air temperature [K]": 300.0,
    "Process temperature [K]": 310.0,
    "Rotational speed [rpm]": 1500.0,
    "Torque [Nm]": 40.0,
    "Tool wear [min]": 50.0,
}

# Add engineered features to match training schema
sample["TempDiff"] = sample["Process temperature [K]"] - sample["Air temperature [K]"]
sample["Torque_x_RPM"] = sample["Torque [Nm]"] * sample["Rotational speed [rpm]"]

x_one = pd.DataFrame([sample])[cat_cols + num_cols]

p = float(calibrated.predict_proba(x_one)[:, 1][0])
p


0.000334001336005344

## Batch scoring + “Top 20 to inspect” list

In [41]:
scored = X_test.copy()
scored["failure_probability"] = proba_cal
scored["y_true"] = y_test.values

top20 = scored.sort_values("failure_probability", ascending=False).head(20)
top20


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],TempDiff,Torque_x_RPM,failure_probability,y_true
5706,L,302.4,311.9,1290,70.0,139,9.5,90300.0,0.995030,1
4661,L,303.3,311.3,1350,48.1,32,8.0,64935.0,0.992908,1
4342,M,301.7,309.8,1284,68.2,111,8.1,87568.8,0.992908,1
8582,M,297.5,308.1,1334,72.0,151,10.6,96048.0,0.992908,1
4179,L,302.4,310.9,1349,46.1,148,8.5,62188.9,0.990196,1
4081,L,302.0,310.4,1336,58.2,110,8.4,77755.2,0.990196,1
4618,M,303.0,311.3,1335,53.6,164,8.3,71556.0,0.990196,1
4283,M,301.8,309.9,1334,61.0,182,8.1,81374.0,0.990196,1
4755,L,303.4,311.7,1295,51.3,51,8.3,66433.5,0.990196,1
4270,L,302.6,310.9,1331,52.0,154,8.3,69212.0,0.990196,1


## Save artifacts for deployment (model + metadata)

In [42]:
ARTIFACT_DIR = "artifacts"
import os
os.makedirs(ARTIFACT_DIR, exist_ok=True)

meta = {
    "target": TARGET,
    "cat_cols": cat_cols,
    "num_cols": num_cols,
    "engineered_features": ["TempDiff", "Torque_x_RPM"],
    "recommended_policy": "topk",
    "topk_k": 20,
    "example_threshold_top20": best_thr_top20,
    "test_pr_auc": float(average_precision_score(y_test, proba_cal)),
}

joblib.dump(calibrated, f"{ARTIFACT_DIR}/pdm_model.joblib")
with open(f"{ARTIFACT_DIR}/pdm_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

meta


{'target': 'Machine failure',
 'cat_cols': ['Type'],
 'num_cols': ['Air temperature [K]',
  'Process temperature [K]',
  'Rotational speed [rpm]',
  'Torque [Nm]',
  'Tool wear [min]',
  'TempDiff',
  'Torque_x_RPM'],
 'engineered_features': ['TempDiff', 'Torque_x_RPM'],
 'recommended_policy': 'topk',
 'topk_k': 20,
 'example_threshold_top20': 0.9908855692071712,
 'test_pr_auc': 0.9069876185020286}